# DSC Logika UI 2026 — Alternative Modeling

This notebook documents and reproduces the v59 alternative experiments.
It uses only local competition data and manually selected numeric features.

Models tested:
- ExtraTrees + HistGradientBoosting.
- XGBoost Lossguide + CatBoost Lossguide.

OOF metrics are diagnostic only. Kaggle public score can differ because the public split is noisy.


## Constraints

- Python only.
- No external data.
- Manual preprocessing and feature engineering only.
- No AutoML or AutoFE.
- Fold-safe feature selection.
- No future records beyond decision time.
- New files stay under `agas/`.
- Do not push to GitHub.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != 'dsc-logika-ui-2026':
    ROOT = Path('/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026')
AGAS = ROOT / 'agas'
OUT = AGAS / 'output'
print(ROOT)


## Reproduce the alternative runs

Set `RUN_TRAINING = True` to execute both training scripts. Outputs stay under `agas/output/`.


In [ ]:
import subprocess

RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run([
        'python3', str(AGAS / 'train_submission_v59_alternative.py'),
        '--folds', '5', '--seed', '2026', '--top-features', '120', '--out-prefix', 'v59alt'
    ], cwd=ROOT, check=True)
    subprocess.run([
        'python3', str(AGAS / 'train_submission_v59_boost.py'),
        '--folds', '5', '--seed', '2026', '--top-features', '180', '--out-prefix', 'v59boost'
    ], cwd=ROOT, check=True)
print('training switch:', RUN_TRAINING)


In [ ]:
for name in ['summary_v59alt.json', 'summary_v59boost.json']:
    path = OUT / name
    if path.exists():
        print(name)
        print(json.dumps(json.loads(path.read_text()), indent=2))


## Interpretation

These alternatives are retained for auditability and future blending research.
They are not selected as the main submission when their OOF KS is below the proven v58ext direction.


In [ ]:
test = pd.read_csv(ROOT / 'data/test_applications.csv')
for path in sorted(OUT.glob('submission_v59*.csv')):
    sub = pd.read_csv(path)
    valid = (
        len(sub) == len(test)
        and sub.application_id.is_unique
        and set(sub.application_id) == set(test.application_id)
        and sub.default_probability.between(0, 1).all()
    )
    print(path.name, 'VALID' if valid else 'INVALID')
